# Controlled Sabotage: Testing Test Quality

## Purpose

Controlled sabotage is a technique to evaluate the quality of your test suite. By intentionally introducing bugs into the code, you can verify that your tests actually detect them.

If your tests pass despite the sabotage, it means your tests are **not adequate**.

---

## Sabotage Scenario 1: Always Return True

### Original Implementation
```python
def add_task(self, title, description=""):
    if not title or not title.strip():
        raise ValueError("Title cannot be empty")
    
    task = {'title': title.strip(), 'description': description.strip()}
    
    try:
        self.storage.save(task)
    except Exception as e:
        raise e
    
    try:
        self.notifier.send(f"New task added: {title}")
    except Exception as e:
        print(f"Notification failed: {e}")
    
    return True
```

### Sabotaged Implementation
```python
def add_task(self, title, description=""):
    # SABOTAGE: Always return True, ignore everything
    return True
```

### Testing the Sabotage

Let's see if our initial tests detect this sabotage.

In [ ]:
# Simulate the sabotaged version
class SabotagedService:
    def __init__(self, storage=None, notifier=None):
        self.storage = storage
        self.notifier = notifier
    
    def add_task(self, title, description=""):
        # SABOTAGE: Always return True, ignore everything
        return True
    
    def get_tasks(self):
        if self.storage:
            return self.storage.get_all()
        return []
    
    def get_notifications(self):
        if self.notifier:
            return self.notifier.get_notifications()
        return []

# Run the initial tests against sabotaged code
def test_add_task_basic_sabotaged():
    service = SabotagedService()
    result = service.add_task("Test task", "Test description")
    assert result is True  # This still passes!
    print("test_add_task_basic: PASSED (but should have FAILED)")

def test_get_tasks_sabotaged():
    service = SabotagedService()
    service.add_task("Task 1")
    tasks = service.get_tasks()
    assert len(tasks) > 0  # This FAILS correctly (no storage)
    print("test_get_tasks: FAILED (correctly detected sabotage)")

def test_notification_sent_sabotaged():
    service = SabotagedService()
    service.add_task("Task 1")
    notifications = service.get_notifications()
    assert len(notifications) > 0  # This FAILS correctly (no notifier)
    print("test_notification_sent: FAILED (correctly detected sabotage)")

# Run tests
try:
    test_add_task_basic_sabotaged()
except AssertionError as e:
    print(f"test_add_task_basic: FAILED - {e}")

try:
    test_get_tasks_sabotaged()
except AssertionError as e:
    print(f"test_get_tasks: FAILED - {e}")

try:
    test_notification_sent_sabotaged()
except AssertionError as e:
    print(f"test_notification_sent: FAILED - {e}")

## Analysis of Results

### What Happened?

- `test_add_task_basic`: **PASSED** despite sabotage ❌
  - This test only checks the return value
  - It doesn't verify that storage or notifier were called

- `test_get_tasks`: **FAILED** correctly ✅
  - This test checks the actual state of storage
  - Since storage was never called, it detected the problem

- `test_notification_sent`: **FAILED** correctly ✅
  - This test checks the actual state of notifier
  - Since notifier was never called, it detected the problem

### Lesson Learned

The first test is **weak** because it only checks the return value without verifying side effects. Good integration tests should:
1. Verify the return value
2. Verify that dependencies were called correctly
3. Verify the state of the system after the operation

## Sabotage Scenario 2: Skip Storage

### Sabotaged Implementation
```python
def add_task(self, title, description=""):
    if not title or not title.strip():
        raise ValueError("Title cannot be empty")
    
    # SABOTAGE: Skip storage, only notify
    # self.storage.save(task)  # Commented out
    
    self.notifier.send(f"New task added: {title}")
    return True
```

In [ ]:
class SabotagedServiceSkipStorage:
    def __init__(self, storage=None, notifier=None):
        self.storage = storage
        self.notifier = notifier
    
    def add_task(self, title, description=""):
        if not title or not title.strip():
            raise ValueError("Title cannot be empty")
        
        # SABOTAGE: Skip storage, only notify
        if self.notifier:
            self.notifier.send(f"New task added: {title}")
        return True
    
    def get_tasks(self):
        if self.storage:
            return self.storage.get_all()
        return []
    
    def get_notifications(self):
        if self.notifier:
            return self.notifier.get_notifications()
        return []

from src.storage import Storage
from src.notifier import Notifier

# Test against this sabotage
storage = Storage()
notifier = Notifier()
service = SabotagedServiceSkipStorage(storage=storage, notifier=notifier)

service.add_task("Test Task")

print(f"Tasks in storage: {len(storage.get_all())}")  # Should be 0 (sabotaged)
print(f"Notifications sent: {len(notifier.get_notifications())}")  # Should be 1

# A good integration test would catch this
assert len(storage.get_all()) == 1, "Storage should have the task"
assert len(notifier.get_notifications()) == 1, "Notifier should have sent notification"

## Improving Tests to Detect Sabotage

### Weak Test (Original)
```python
def test_add_task_basic():
    service = Service()
    result = service.add_task("Test task", "Test description")
    assert result is True  # Only checks return value
```

### Strong Test (Improved)
```python
def test_add_task_integration():
    service = Service()
    result = service.add_task("Test task", "Test description")
    
    # Check return value
    assert result is True
    
    # Check that storage was actually used
    tasks = service.get_tasks()
    assert len(tasks) == 1
    assert tasks[0]['title'] == "Test task"
    
    # Check that notifier was actually used
    notifications = service.get_notifications()
    assert len(notifications) == 1
    assert "Test task" in notifications[0]
```

### Even Stronger Test (With Mocks)
```python
def test_add_task_with_mocks():
    storage_mock = Mock()
    notifier_mock = Mock()
    service = Service(storage=storage_mock, notifier=notifier_mock)
    
    result = service.add_task("Test task", "Test description")
    
    # Verify the methods were called with correct arguments
    storage_mock.save.assert_called_once_with({'title': 'Test task', 'description': 'Test description'})
    notifier_mock.send.assert_called_once_with("New task added: Test task")
```

## Key Takeaways

1. **Tests that only check return values are weak** - They can be fooled by sabotaged code

2. **Integration tests must verify side effects** - Check that dependencies were actually called

3. **Use mocks to verify interactions** - Mocks can assert that methods were called with specific arguments

4. **Controlled sabotage reveals test weaknesses** - If sabotage doesn't break tests, the tests need improvement

5. **Test the system state, not just outputs** - Verify the actual state of storage, database, etc.

---

## Exercise

Try sabotaging the code in different ways and see if your tests catch them:

1. Make the notifier always fail
2. Make storage raise an exception
3. Return False instead of True
4. Modify the task data before saving

For each sabotage, ask yourself: **Did my tests detect this? If not, how can I improve them?**